In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score


In [2]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

In [3]:
# csv파일이 데이터가 큰데 샘플을 10000개 행만 뽑아서 저장
df = pd.read_csv('../data/감자_월차_학습용.csv', encoding='cp949')

# 등급코드 제거
df = df.drop(columns=['등급코드'], errors='ignore')

# y, X 분리
y = df['평균단가(원)']
X = df.drop(columns=['평균단가(원)'])

# 날짜 처리
X['week_start'] = pd.to_datetime(X['week_start'])
X['year'] = X['week_start'].dt.year


In [4]:
# 학습용: 2024년까지
X_train = X[X['year'] < 2025].drop(columns=['week_start'])
y_train = y[X['year'] < 2025]

# 예측용: 2025년
X_test = X[X['year'] == 2025].drop(columns=['week_start'])
y_test = y[X['year'] == 2025]

print(f"학습 데이터 크기: {X_train.shape}")
print(f"예측 데이터 크기: {X_test.shape}")


학습 데이터 크기: (152432, 197)
예측 데이터 크기: (7581, 197)


In [5]:
param_distributions = {
    'n_estimators': [300, 500, 800],
    'max_depth': [8, 10, 12, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 0.5, 0.8],
    'min_impurity_decrease': [0.0, 0.01],
    'bootstrap': [True]
}


In [6]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)


Fitting 3 folds for each of 20 candidates, totalling 60 fits


,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'bootstrap': [True], 'max_depth': [8, 10, ...], 'max_features': ['sqrt', 0.5, ...], 'min_impurity_decrease': [0.0, 0.01], ...}"
,n_iter,20
,scoring,'neg_root_mean_squared_error'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
# 최적 모델 추출
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

# 평가 지표 출력
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("✅ 최적 하이퍼파라미터:", random_search.best_params_)
print(f"📉 2025년 RMSE: {rmse:.2f}")
print(f"📈 2025년 R² Score: {r2:.4f}")


✅ 최적 하이퍼파라미터: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'min_impurity_decrease': 0.01, 'max_features': 0.5, 'max_depth': 8, 'bootstrap': True}
📉 2025년 RMSE: 397.87
📈 2025년 R² Score: 0.7330


In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

X['week_start'] = pd.to_datetime(X['week_start'])
X['year'] = X['week_start'].dt.year
X['week'] = X['week_start'].dt.isocalendar().week.astype(int)

# 주기형 특성 생성
X['week_sin'] = np.sin(2 * np.pi * X['week'] / 52)
X['week_cos'] = np.cos(2 * np.pi * X['week'] / 52)
X = X.drop(columns=['week'])  # 기존 week는 제거

# 최신 연도 자동 분리
latest_year = X['year'].max()
print(f" 예측 연도: {latest_year}")

# 학습용: 2024년까지
X_train = X[X['year'] < 2025].drop(columns=['week_start'])
y_train = y[X['year'] < 2025]

# 예측용: 2025년
X_test = X[X['year'] == 2025].drop(columns=['week_start'])
y_test = y[X['year'] == 2025]


print(f" 학습 데이터: {X_train.shape}, 예측 데이터: {X_test.shape}")

# 실무기준 정밀 하이퍼파라미터 설정
param_distributions = {
    'n_estimators': [500, 800, 1200, 1600],
    'max_depth': [15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 0.2, 0.3],
    'min_impurity_decrease': [0.0, 0.0005, 0.001],
    'bootstrap': [True]
}

# 반복 학습 조건
TARGET_RMSE = 300
MAX_TRIALS = 5

trial = 0
best_score = float('inf')
best_r2 = -1
best_model = None

while trial < MAX_TRIALS:
    print(f"ㅋㅋ\ 튜닝 반복 {trial+1}/{MAX_TRIALS}")

    search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
        param_distributions=param_distributions,
        n_iter=20,
        scoring='neg_root_mean_squared_error',
        cv=3,
        verbose=1,
        random_state=trial + 42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)
    model = search.best_estimator_
    pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"➡️ RMSE: {rmse:.2f} / R²: {r2:.4f}")

    if rmse < TARGET_RMSE and r2 >= 0.75:
        print(" 점수 기준 만족! 반복 종료")
        best_model = model
        break

    if rmse < best_score:
        best_score = rmse
        best_r2 = r2
        best_model = model

    trial += 1

# 최종 결과 출력
print("\n🏁 최종 모델 성능:")
print(f"📉 RMSE: {best_score:.2f}")
print(f"📈 R² : {best_r2:.4f}")


 예측 연도: 2025
 학습 데이터: (152432, 198), 예측 데이터: (7581, 198)
ㅋㅋ\ 튜닝 반복 1/5
Fitting 3 folds for each of 20 candidates, totalling 60 fits
➡️ RMSE: 395.06 / R²: 0.7367
ㅋㅋ\ 튜닝 반복 2/5
Fitting 3 folds for each of 20 candidates, totalling 60 fits
➡️ RMSE: 405.44 / R²: 0.7227
ㅋㅋ\ 튜닝 반복 3/5
Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

In [9]:
# plt.figure(figsize=(8, 5))
# plt.plot(range(1, len(rmse_scores)+1), rmse_scores, marker='o', linestyle='--')
# plt.title("TimeSeriesSplit 별 RMSE")
# plt.xlabel("Fold")
# plt.ylabel("RMSE")
# plt.grid(True)
# plt.tight_layout()
# plt.show()
